In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, QuantoConfig
import torch

model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    # quantization_config=QuantoConfig(weights="int8"),
    torch_dtype=torch.float16
    ).to(device)
max_new_tokens = 512

/Users/rojankarki/Projects/watermark-llm-code-quality/watermark-llm-codebase/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 339/339 [00:16<00:00, 20.57it/s]


In [2]:
print(model.dtype)                          # e.g. torch.float16
print(next(model.parameters()).dtype)       # cross-check, same result
print(model.get_memory_footprint() / 1e9, "GB")

torch.float16
torch.float16
15.231233536 GB


In [3]:
import json

def read_json(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        dataset = json.load(f)
    return dataset

def write_json(filename, json_data):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(json_data, f, indent=2, ensure_ascii=False)

# Check Prepare Query and Response using a sample from dataset.json

In [3]:
def prepare_query(data, language = "python"):
    language = "python" 
    hint = data["hints"].get(language, "").strip()
    hint_line = f"Hint: {hint}\n" if hint else ""

    query = f"""Write a {language} code for the following task description: {data['prompt_description']}
{hint_line} """
    return query

# MBPP


In [4]:
import json
import random
random.seed(100)
sample_size = 200
full_dataset_path = '../data/sanitized-mbpp.json'
sampled_path = '../data/sampled-sanitized-mbpp.json'

In [38]:
# DONT RERUN

# san_dataset = read_json(full_dataset_path)
# print(f"Total entries: {len(san_dataset)}")

# dataset = random.sample(san_dataset, sample_size)
# print(f"Sampled size: {len(dataset)}")
# write_json(sampled_path, dataset)

# print(json.dumps(dataset[-1],indent=2))

In [5]:
# Load the dataset
dataset = read_json(sampled_path)

print(f"Total entries: {len(dataset)}")
print(json.dumps(dataset[0],indent=2))

Total entries: 200
{
  "source_file": "Mike's Copy of Benchmark Questions Verification V2.ipynb",
  "task_id": 126,
  "prompt": "Write a python function to find the sum of common divisors of two given numbers.",
  "code": "def sum(a,b): \n    sum = 0\n    for i in range (1,min(a,b)): \n        if (a % i == 0 and b % i == 0): \n            sum += i \n    return sum",
  "test_imports": [],
  "test_list": [
    "assert sum(10,15) == 6",
    "assert sum(100,150) == 93",
    "assert sum(4,6) == 3"
  ]
}


In [ ]:
def prepare_mbpp_query_one_shot(data, sample, language = "python"):
    requirement = f"""
Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.
"""
    one_shot_example = f"""Write a {language} code for the following task description: {sample['text']}
{requirement}
### EXAMPLE
Test cases: {sample['test_list']}
Output: ```{language}
{sample['code']}
```"""
    query = f"""{one_shot_example}
### TARGET
Now, write a {language} code for the following task description: {data['text']}
{requirement}
Test cases: {data['test_list']}
Output: 
"""
    return query

In [6]:
def prepare_mbpp_query(data, language = "python"):
    query = f"""Write a {language} code for the following task description: {data['prompt']}
Your code should pass these test cases: {data['test_list']}
Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.
    """
    return query

In [7]:
query = prepare_mbpp_query(dataset[0] )
print(query)

Write a python code for the following task description: Write a python function to find the sum of common divisors of two given numbers.
Your code should pass these test cases: ['assert sum(10,15) == 6', 'assert sum(100,150) == 93', 'assert sum(4,6) == 3']
Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.
    


In [8]:
messages = [
    {"role": "user", "content": query}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

# inputs = tokenizer(query, return_tensors="pt", add_special_tokens = True).to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.1,
    )

# response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
response = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0]

print(response)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a python code for the following task description: Write a python function to find the sum of common divisors of two given numbers.
Your code should pass these test cases: ['assert sum(10,15) == 6', 'assert sum(100,150) == 93', 'assert sum(4,6) == 3']
Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.
    
assistant
```python
def sum(n,m): 
    s = 0
    for i in range(1,min(n,m)+1): 
        if (n%i==0 and m%i==0): 
            s += i 
    return s 
```


# MARKLLM Framework

In [11]:
from watermark.auto_watermark import AutoWatermark
from utils.transformers_config import TransformersConfig

# Transformers config
transformers_config = TransformersConfig(model=model,
                                         tokenizer=tokenizer,
                                         device=device,
                                         max_new_tokens=max_new_tokens,
                                         min_length=230,
                                         do_sample=True,
                                         no_repeat_ngram_size=4,
                                        #  temperature=0.1,
                                         )


In [12]:
# Load watermark algorithm
myWatermark = AutoWatermark.load('SynthID', 
                                 algorithm_config='config/SynthID.json',
                                 transformers_config=transformers_config)

In [22]:
for j in range(2):
    filename = f"result-sanitized-mbpp-iter-{j}.json"
    result = []
    for i, data in enumerate(dataset):
        print(f"Processing index: {i}")
        query = prepare_mbpp_query(data)
        messages = [
            {"role": "user", "content": query}
        ]
        query = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        watermarked_text = myWatermark.generate_watermarked_text(query)
        unwatermarked_text = myWatermark.generate_unwatermarked_text(query)
        # detect_result_watermarked_text = myWatermark.detect_watermark(watermarked_text)
        # detect_result_unwatermarked_text = myWatermark.detect_watermark(unwatermarked_text)
        result.append({
            **data,
            "outputs": {
                "watermarked": {
                    "content": watermarked_text,
                    # **detect_result_watermarked_text,
                },
                "unwatermarked":{
                    "content": unwatermarked_text,
                    # **detect_result_unwatermarked_text,
                }
            }
        })
    print(json.dumps(result, indent=2))
    write_json(filename, result)    


Processing index: 0
Processing index: 1
Processing index: 2
Processing index: 3
Processing index: 4
Processing index: 5
Processing index: 6
Processing index: 7
Processing index: 8
Processing index: 9
Processing index: 10
Processing index: 11
Processing index: 12
Processing index: 13
Processing index: 14
Processing index: 15
Processing index: 16
Processing index: 17
Processing index: 18
Processing index: 19
Processing index: 20
Processing index: 21
Processing index: 22
Processing index: 23
Processing index: 24
Processing index: 25
Processing index: 26
Processing index: 27
Processing index: 28
Processing index: 29
Processing index: 30
Processing index: 31
Processing index: 32
Processing index: 33
Processing index: 34
Processing index: 35
Processing index: 36
Processing index: 37
Processing index: 38
Processing index: 39
Processing index: 40
Processing index: 41
Processing index: 42
Processing index: 43
Processing index: 44
Processing index: 45
Processing index: 46
Processing index: 47
Pr

In [16]:
result = []
for i, data in enumerate(dataset):
    print(f"Processing index: {i}")
    query = prepare_mbpp_query(data)
    messages = [
        {"role": "user", "content": query}
    ]
    query = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    watermarked_text = myWatermark.generate_watermarked_text(query)
    unwatermarked_text = myWatermark.generate_unwatermarked_text(query)
    # detect_result_watermarked_text = myWatermark.detect_watermark(watermarked_text)
    # detect_result_unwatermarked_text = myWatermark.detect_watermark(unwatermarked_text)
    result.append({
        **data,
        "outputs": {
            "watermarked": {
                "content": watermarked_text,
                # **detect_result_watermarked_text,
            },
            "unwatermarked":{
                "content": unwatermarked_text,
                # **detect_result_unwatermarked_text,
            }
        }
    })

print(json.dumps(result, indent=2))


Processing index: 0
Processing index: 1
Processing index: 2
Processing index: 3
Processing index: 4
Processing index: 5
Processing index: 6
Processing index: 7
Processing index: 8
Processing index: 9
Processing index: 10
Processing index: 11
Processing index: 12
Processing index: 13
Processing index: 14
Processing index: 15
Processing index: 16
Processing index: 17
Processing index: 18
Processing index: 19
Processing index: 20
Processing index: 21
Processing index: 22
Processing index: 23
Processing index: 24
Processing index: 25
Processing index: 26
Processing index: 27
Processing index: 28
Processing index: 29
Processing index: 30
Processing index: 31
Processing index: 32
Processing index: 33
Processing index: 34
Processing index: 35
Processing index: 36
Processing index: 37
Processing index: 38
Processing index: 39
Processing index: 40
Processing index: 41
Processing index: 42
Processing index: 43
Processing index: 44
Processing index: 45
Processing index: 46
Processing index: 47
Pr

In [17]:
write_json('../data/result-sanitized-mbpp-it-3.json', result)

In [15]:
len(result)

200

In [ ]:
for i in range(2):
    result = []
    for data in dataset:
        query = prepare_query(data)
        messages = [
            {"role": "user", "content": query}
        ]

        query = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        watermarked_text = myWatermark.generate_watermarked_text(query)
        unwatermarked_text = myWatermark.generate_unwatermarked_text(query)
        detect_result_watermarked_text = myWatermark.detect_watermark(watermarked_text)
        detect_result_unwatermarked_text = myWatermark.detect_watermark(unwatermarked_text)
        result.append({
            **data,
            "outputs": {
                "watermarked": {
                    "content": watermarked_text,
                    **detect_result_watermarked_text,
                },
                "unwatermarked":{
                    "content": unwatermarked_text,
                    **detect_result_unwatermarked_text,
                }
            }
        })
    filename = f"result-{i}.json"
    write_json(filename, result)


In [41]:
print(result[0]['outputs']['watermarked']['content'])

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a python code for the following task description: Write a python function to find the sum of common divisors of two given numbers.
Your code should pass these test cases: ['assert sum(10,15) == 6', 'assert sum(100,150) == 93', 'assert sum(4,6) == 3']
Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.
    
assistant
```python
def sum(n, m):
    s = 0
    i = 1
    while (i <= n and i <= m):
        if (n=fopenm and m % i == 0):
            s += i
        i += 1
return s
``` 
Please note that there was an error in your requirements where you mentioned `_weak_` instead of the correct mathematical operator `%` for modulus. I've corrected this in the provided solution. Also, I've removed the unnecessary `fopenm` which seems to be a typo or mistake in your original question.


In [43]:
print(result[0]['outputs']['unwatermarked']['content'])

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a python code for the following task description: Write a python function to find the sum of common divisors of two given numbers.
Your code should pass these test cases: ['assert sum(10,15) == 6', 'assert sum(100,150) == 93', 'assert sum(4,6) == 3']
Requirements:
- Return only the source code.
- Do not include explanations, comments or Markdown.
    
assistant
```python
def sum(num1,num2):
    total = 0
    if num1 < num2:
        for i in range(1,num1+1):
            if (num1 % i == 0 and num2 % i ==0):
                total += i
    else:
        for j in range(2,num2+1):
             if (num2 % j == 0):
                 total +=j
    return total
```  \\\\
```python
```


# MISC

In [29]:
import re

def extract_code_block(output: str, language: str = "python") -> str:
    pattern = rf"```{language}\s*\n(.*?)```"
    match = re.search(pattern, output, re.DOTALL)
    if match:
        return match.group(1).strip()
    return ""

In [37]:
watermarked_text_code = extract_code_block(result[0]['outputs']['watermarked']['content'])
print(watermarked_text_code)

import os
import threading
from queue import Queue

def process_file(file_path, output_queue):
    with open겥


In [31]:
unwatermarked_text_code = extract_code_block(unwatermarked_text)
print(unwatermarked_text_code)

def reverse_five_or_more(s):
    return ' '.join(word[::-1] if len(word) >= 5 else word for word in s.split())

# Test cases
print(reverse_five_or_more("Hey fellow warriors"))  # Output: "Hey wollef sroirraw"
print(reverse_five_or_more("This is a test"))       # Output: "This is a test"
print(reverse_five_or_more("This is another test")) # Output: "This is rehtona test"


In [33]:
def strip_prompt(output: str, query: str) -> str:
    if output.startswith(query):
        return output[len(query):].strip()
    # fallback: find query anywhere and take everything after it
    idx = output.find(query)
    if idx != -1:
        return output[idx + len(query):].strip()
    return output.strip()

unwatermarked = strip_prompt(unwatermarked_text, query)
print(unwatermarked)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a python code for the following task description: Write a function that takes in a string of one or more words, and returns the same string, but with all words that have five or more letters reversed (Just like the name of this Kata). Strings passed in will consist of only letters and spaces. Spaces will be included only when more than one word is present.

Examples:

"Hey fellow warriors"  --> "Hey wollef sroirraw" 
"This is a test        --> "This is a test" 
"This is another test" --> "This is rehtona test"
Output only the raw python code — no explanations, no markdown code fences, no preamble or postamble. The code must be complete and directly executable when saved to a file 
assistant
```python
def reverse_five_or_more(s):
    return ' '.join(word[::-1] if len(word) >= 5 else word for word in s.split())

# Test cases
print(reverse_five_or_more("Hey fellow warriors"))  # Output: "Hey wollef sroi

# VISUALIZATION

In [34]:
from visualize.font_settings import FontSettings
from visualize.visualizer import DiscreteVisualizer
from visualize.legend_settings import DiscreteLegendSettings
from visualize.page_layout_settings import PageLayoutSettings
from visualize.color_scheme import ColorSchemeForDiscreteVisualization

In [35]:
watermarked_data = myWatermark.get_data_for_visualization(watermarked_text)
unwatermarked_data = myWatermark.get_data_for_visualization(unwatermarked_text)

# Init visualizer
visualizer = DiscreteVisualizer(color_scheme=ColorSchemeForDiscreteVisualization(),
                                font_settings=FontSettings(), 
                                page_layout_settings=PageLayoutSettings(),
                                legend_settings=DiscreteLegendSettings())
# Visualize
watermarked_img = visualizer.visualize(data=watermarked_data, 
                                       show_text=True, 
                                       visualize_weight=True, 
                                       display_legend=True)

unwatermarked_img = visualizer.visualize(data=unwatermarked_data,
                                         show_text=True, 
                                         visualize_weight=True, 
                                         display_legend=True)

In [36]:
from PIL import Image

def side_by_side(img1: Image.Image, img2: Image.Image) -> Image.Image:
    w = img1.width + img2.width
    h = max(img1.height, img2.height)
    combined = Image.new("RGB", (w, h), (255, 255, 255))
    combined.paste(img1, (0, 0))
    combined.paste(img2, (img1.width, 0))
    return combined

combined_img = side_by_side(watermarked_img, unwatermarked_img)
combined_img.show()  # opens in default image viewer